# 🧪 NIDS Model Training & Evaluation Experiments

This notebook documents the training pipeline for the Network Intrusion Detection System (NIDS). 
It includes data preprocessing, model training (Random Forest & XGBoost), performance evaluation, and saving trained models.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# إضافة الجذر للتعرف على مجلد src
sys.path.append(os.path.abspath(os.path.join("..")))

from src.data.preprocessing import DataPreprocessor

# 1. تحميل عينة البيانات
data_path = os.path.join("..", "data", "sample", "sample_network_traffic.csv")
df_raw = pd.read_csv(data_path)

# 2. تنظيف البيانات ومعالجتها
preprocessor = DataPreprocessor()
X_processed, y_processed = preprocessor.fit_transform(df_raw)

print(f"✅ Data Loaded & Processed Successfully!")
print(f"Features shape: {X_processed.shape}")
print(f"Target distribution:\n{y_processed.value_counts()}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# 1. تقسيم البيانات (80% تدريب - 20% اختبار)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_processed, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_processed
)

# 2. تدريب نموذج Random Forest
print("🚀 Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 3. تدريب نموذج XGBoost
print("🚀 Training XGBoost...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train, y_train)

print("✨ Both models trained successfully!")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# توقع النتائج
rf_preds = rf_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

print("=" * 40)
print(f"🌲 Random Forest Accuracy: {accuracy_score(y_test, rf_preds) * 100:.2f}%")
print("=" * 40)
print(classification_report(y_test, rf_preds))

print("=" * 40)
print(f"⚡ XGBoost Accuracy: {accuracy_score(y_test, xgb_preds) * 100:.2f}%")
print("=" * 40)
print(classification_report(y_test, xgb_preds))

In [ ]:
import joblib

# إنشاء مجلد models إن لم يكن موجوداً
models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)

# حفظ الملفات
joblib.dump(rf_model, os.path.join(models_dir, "random_forest.pkl"))
joblib.dump(xgb_model, os.path.join(models_dir, "xgboost.pkl"))

print(f"💾 Models successfully saved to '{models_dir}/'")